# Demonstration

The following cell blocks are for exploring the data in the tidywigits parquet output directory.

Set up the Python environment e.g.

```
conda create -n oncoglue python=3.13
conda activate oncoglue
```

In [ ]:
# install the dependencies
! pip install duckdb pyarrow pandas matplotlib

In [58]:
! ls "../tidywigits-operator/data/output/tidywigits/0.0.7.9005/"

2025091002d1f664 202509100539df0e


In [ ]:
! ls "../tidywigits-operator/data/output/tidywigits/0.0.7.9005/2025091002d1f664/"

In [60]:
import duckdb
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

In [61]:
BASE = "../tidywigits-operator/data/output/tidywigits/0.0.7.9005/"

In [62]:
tbl_metadata = f"{BASE}/*/metadata.parquet"
tbl_alignments_dupfreq = f"{BASE}/*/*_purple_qc.parquet"
tbl_alignments_dupfreq = f"{BASE}/*/*_alignments_dupfreq.parquet"
tbl_neo_predictions = f"{BASE}/*/*_neo_predictions.parquet"

In [ ]:
# https://duckdb.org/docs/stable/data/multiple_files/overview
duckdb.sql('select distinct input_id, input_prefix from read_parquet("../tidywigits-operator/data/output/tidywigits/0.0.7.9005/*/*_purple_qc.parquet") limit 10')

In [84]:
duckdb.sql(f'select input_id, pkg_versions from read_parquet("{tbl_metadata}")')

┌──────────────────┬──────────────────────────────────────────────────────────────────────────────────────┐
│     input_id     │                                     pkg_versions                                     │
│     varchar      │                     struct("name" varchar, "version" varchar)[]                      │
├──────────────────┼──────────────────────────────────────────────────────────────────────────────────────┤
│ 2025091002d1f664 │ [{'name': nemo, 'version': 0.0.3.9022}, {'name': tidywigits, 'version': 0.0.7.9005}] │
│ 202509100539df0e │ [{'name': nemo, 'version': 0.0.3.9022}, {'name': tidywigits, 'version': 0.0.7.9005}] │
└──────────────────┴──────────────────────────────────────────────────────────────────────────────────────┘

In [64]:
# check parquet compression
duckdb.sql('select distinct row_group_id, compression from parquet_metadata("../tidywigits-operator/data/output/tidywigits/0.0.7.9005/*/metadata.parquet")')

┌──────────────┬─────────────┐
│ row_group_id │ compression │
│    int64     │   varchar   │
├──────────────┼─────────────┤
│            0 │ SNAPPY      │
└──────────────┴─────────────┘

In [65]:
r = duckdb.read_parquet(tbl_alignments_dupfreq)

In [66]:
df = duckdb.sql('select * from r limit 10')

In [ ]:
print(df)

In [68]:
duckdb.sql('select input_id, output_id, version as purple_version from read_parquet("../tidywigits-operator/data/output/tidywigits/0.0.7.9005/*/*_purple_version.parquet") group by input_id, output_id, version order by version')

┌──────────────────┬────────────────────────────┬────────────────┐
│     input_id     │         output_id          │ purple_version │
│     varchar      │          varchar           │    varchar     │
├──────────────────┼────────────────────────────┼────────────────┤
│ 202509100539df0e │ 01KTT9GC23VNCR4K2PMG7735AE │ 4.2            │
│ 2025091002d1f664 │ 01KTT9FP9ATDM51MHYAHJW28S3 │ 4.2            │
└──────────────────┴────────────────────────────┴────────────────┘

In [69]:
duckdb.sql('select input_id, output_id, version as linx_version from read_parquet("../tidywigits-operator/data/output/tidywigits/0.0.7.9005/*/*_linx_version.parquet") group by input_id, output_id, version order by version')

┌──────────────────┬────────────────────────────┬──────────────┐
│     input_id     │         output_id          │ linx_version │
│     varchar      │          varchar           │   varchar    │
├──────────────────┼────────────────────────────┼──────────────┤
│ 202509100539df0e │ 01KTT9GC23VNCR4K2PMG7735AE │ 2.1          │
│ 2025091002d1f664 │ 01KTT9FP9ATDM51MHYAHJW28S3 │ 2.1          │
└──────────────────┴────────────────────────────┴──────────────┘

In [ ]:
duckdb.sql(f'''
    select input_id, input_prefix, ne_id, variant_type, variant_info, gene_name, peptide_count
    from read_parquet("{tbl_neo_predictions}")
    where peptide_count > 50
''')

In [ ]:
df1 = duckdb.sql(f'select * from read_parquet("{tbl_neo_predictions}")').to_df()
x_col = "gene_name"
y_col = "peptide_count"
df1.plot(x=x_col, y=y_col, kind="line", figsize=(10, 4), title=f"{y_col} by {x_col}")
plt.tight_layout()
plt.show()

In [ ]:
duckdb.sql(f'select * from parquet_scan("{tbl_alignments_dupfreq}")')

In [ ]:
duckdb.sql(f'''
    select
        input_id,
        input_prefix,
        output_id,
        avg(cast(duplicate_read_count as double)) as average_duplicate_read_count,
        avg(frequency)
    from
        parquet_scan("{tbl_alignments_dupfreq}")
    group by input_id, input_prefix, output_id
''')

In [74]:
df = duckdb.sql(f'select * from parquet_scan("{tbl_alignments_dupfreq}")')

In [ ]:
df.describe()

In [76]:
duckdb.sql(f'describe "{tbl_alignments_dupfreq}"')

┌──────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name      │ column_type │  null   │   key   │ default │  extra  │
│       varchar        │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ input_id             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ input_prefix         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ output_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ duplicate_read_count │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ frequency            │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
└──────────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

In [77]:
# print arrow schema using pyarrow
schema = pq.read_schema(f"{BASE}/2025091002d1f664/metadata.parquet")
print(schema)

input_id: string
output_id: string
input_dirs: list<element: string>
  child 0, element: string
output_dir: string
pkg_versions: list<element: struct<name: string, version: string>>
  child 0, element: struct<name: string, version: string>
      child 0, name: string
      child 1, version: string
files: list<element: struct<tbl: string, prefix: string, fout: string, fin: string>>
  child 0, element: struct<tbl: string, prefix: string, fout: string, fin: string>
      child 0, tbl: string
      child 1, prefix: string
      child 2, fout: string
      child 3, fin: string


In [92]:
# create temporary table query technique
duckdb.sql('drop table if exists tmp_alignments_dupfreq')

# note `... with no data` statement, this causes reading the table structure only
duckdb.sql(f'create table tmp_alignments_dupfreq as select * from "{tbl_alignments_dupfreq}" with no data')

In [93]:
duckdb.sql('describe tmp_alignments_dupfreq')

┌──────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name      │ column_type │  null   │   key   │ default │  extra  │
│       varchar        │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ input_id             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ input_prefix         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ output_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ duplicate_read_count │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ frequency            │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
└──────────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

In [95]:
# should output zero rows as no data - structure only
duckdb.sql('select * from tmp_alignments_dupfreq')

┌──────────┬──────────────┬───────────┬──────────────────────┬───────────┐
│ input_id │ input_prefix │ output_id │ duplicate_read_count │ frequency │
│ varchar  │   varchar    │  varchar  │       varchar        │  double   │
└──────────┴──────────────┴───────────┴──────────────────────┴───────────┘
                                  0 rows                                

In [81]:
arrow_tbl = pq.read_table(f"{BASE}/2025091002d1f664/metadata.parquet")
metadata = arrow_tbl.to_pydict()
assert len(metadata['files']) == 1 # `files` attribute is in nested list of list of dicts
files = metadata['files'][0]

tables = set()

for file in files:
    tables.add(file['tbl'])
print(len(tables))
print("-" * 32)

for tbl in sorted(tables):
    print(tbl)

75
--------------------------------
alignments_dupfreq
amber_bafpcf
amber_contaminationtsv
amber_homozygousregion
amber_qc
amber_version
bamtools_coverage
bamtools_exoncvg
bamtools_flagstats
bamtools_fraglength
bamtools_genecvg_cvg
bamtools_genecvg_genes
bamtools_partitionstats
bamtools_summary_dp
bamtools_summary_stats
chord_prediction
chord_signatures
cider_blastn
cider_locusstats
cider_vdj
cobalt_gcmed_buckets
cobalt_gcmed_sample
cobalt_ratiomed
cobalt_ratiopcf
cobalt_version
cuppa_predsum
cuppa_visdata
esvee_assemblealignment
esvee_assembleassembly
esvee_assemblebreakend
esvee_assemblephased
esvee_prepdiscstats
esvee_prepfraglen
esvee_prepjunction
lilac_qc
lilac_summary
linx_breakends
linx_clusters
linx_drivercatalog
linx_drivers
linx_fusions
linx_links
linx_neoepitope
linx_svs
linx_version
linx_viscn
linx_visfusion
linx_visgeneexon
linx_visproteindomain
linx_vissegments
linx_vissvdata
neo_candidates
neo_predictions
peach_events
peach_geneevents
peach_haplotypesall
peach_haplotypes

In [82]:
# print out the schema of each table
for tbl in sorted(tables):
    print(tbl)
    duckdb.sql(f'describe "{BASE}/*/*_{tbl}.parquet"').show(max_rows=1000)

alignments_dupfreq
┌──────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name      │ column_type │  null   │   key   │ default │  extra  │
│       varchar        │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ input_id             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ input_prefix         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ output_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ duplicate_read_count │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ frequency            │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
└──────────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

amber_bafpcf
┌──────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name  │ column_type │  null   │   key   │ default │  extra  │
│   varchar    │   varchar   │ varc